In [1]:
!pip install -U --force-reinstall --no-deps transformers
!pip install -U huggingface_hub tokenizers safetensors krippendorff 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 34.5 MB/s  0:00:00m0:00:010:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 7.2 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0━━━━━━━━━━━━━ 1/2 [huggingface_hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [huggingface_hub] [huggingface_hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but 

In [3]:
import glob
import numpy as np
import pandas as pd
from datasets import Dataset
from scipy.stats import spearmanr, pearsonr
from transformers import (AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification, pipeline, TrainingArguments, Trainer)
import krippendorff
import torch
from safetensors.torch import load_file
import shutil

In [4]:
###############################################################################
# pre-trained model

from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification, pipeline

name = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(name)

# For the fill-mask demo
mlm_model = AutoModelForMaskedLM.from_pretrained(name)
fill = pipeline("fill-mask", model=mlm_model, tokenizer=tokenizer)
for r in fill(f"Sveriges huvudstad är {tokenizer.mask_token}."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")

# For the ABSAbank-Imm benchmark
reg_model = AutoModelForSequenceClassification.from_pretrained(
    name, num_labels=1, problem_type="regression")

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

   Stockholm  0.361
 Helsingfors  0.094
     Uppsala  0.039
        Lund  0.033
     Tallinn  0.022


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
for r in fill(f"Du kör inne i en tätort. Det finns inga hastighetsskyltar. Hastighet som gäller är {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")


         100  0.013
         180  0.012
         160  0.012
         220  0.011
          50  0.011


In [9]:
for r in fill(f"Du kör på en väg utanför tätort utan hastighetsskyltar. Högsta tillåtna hastighet är {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")


         100  0.016
         180  0.015
         160  0.015
          50  0.013
         200  0.011


In [11]:
for r in fill(f"Om det inte finns några hastighetsskyltar är högsta tillåtna hastighet inom en tätort {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")

         100  0.038
          50  0.019
        1000  0.017
          60  0.016
         180  0.014


In [12]:
for r in fill(f"Om det inte finns några hastighetsskyltar är högsta tillåtna hastighet på en väg utanför tätort {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")

         100  0.026
        1000  0.017
          50  0.016
          60  0.014
         500  0.012


In [13]:
for r in fill(f"På en väg inne i en tätort utan hastighetsskyltar får du köra högst {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")

         100  0.036
         180  0.028
         160  0.024
          50  0.020
         150  0.019


In [14]:
for r in fill(f"På en väg utanför tätort utan hastighetsskyltar får du köra högst {tokenizer.mask_token} kmh."):
    print(f"{r['token_str']:>12}  {r['score']:.3f}")

         100  0.027
         180  0.024
         160  0.022
          50  0.017
         150  0.015


In [4]:
for name, p in reg_model.named_parameters():
    print(name, tuple(p.shape))

distilbert.embeddings.word_embeddings.weight (119547, 768)
distilbert.embeddings.position_embeddings.weight (512, 768)
distilbert.embeddings.LayerNorm.weight (768,)
distilbert.embeddings.LayerNorm.bias (768,)
distilbert.transformer.layer.0.attention.q_lin.weight (768, 768)
distilbert.transformer.layer.0.attention.q_lin.bias (768,)
distilbert.transformer.layer.0.attention.k_lin.weight (768, 768)
distilbert.transformer.layer.0.attention.k_lin.bias (768,)
distilbert.transformer.layer.0.attention.v_lin.weight (768, 768)
distilbert.transformer.layer.0.attention.v_lin.bias (768,)
distilbert.transformer.layer.0.attention.out_lin.weight (768, 768)
distilbert.transformer.layer.0.attention.out_lin.bias (768,)
distilbert.transformer.layer.0.sa_layer_norm.weight (768,)
distilbert.transformer.layer.0.sa_layer_norm.bias (768,)
distilbert.transformer.layer.0.ffn.lin1.weight (3072, 768)
distilbert.transformer.layer.0.ffn.lin1.bias (3072,)
distilbert.transformer.layer.0.ffn.lin2.weight (768, 3072)
dist

The original Swedish ABSAbank contains two layers of annotation: one at token level and one at text level. At the text-level, the annotation consisted of two sublayers: paragraph-level and document-level annotation. Only the paragraph-level annotation is preserved in Absabank-IMM. When creating the original ABSAbank, the annotators had to label every paragraph whose subject matter was immigration (and only those) with a sentiment value on the scale from 1 (very negative) to 5 (very positive).

In [24]:
###############################################################################
# absa data download

def load(split):
    f = glob.glob(f"/kaggle/input/datasets/leejollans/absabank//**/*{split}*.tsv", recursive=True)[0]
    df = pd.read_csv(f, sep="\t", quoting=3) 
    return df[["id", "text", "label"]].astype({"label": float})

train, dev, test = load("train"), load("dev"), load("test")
print('N training', len(train), 'N development', len(dev), 'N test', len(test)); print(train.head())

def prep(df,n):
    ds = Dataset.from_pandas(df.rename(columns={"label": "labels"}).sample(n=n))
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=256),batched=True)

tr, dv, te = prep(train,128), prep(dev,32), prep(test,32)



N training 3898 N development 487 N test 487
                             id  \
0   z02679_flashback-50294563_3   
1   z02679_flashback-50294563_5   
2   z01476_flashback-56524490_1   
3   z01476_flashback-56524490_3   
4  z01476_flashback-56524490_10   

                                                text  label  
0  Förklarar regeringen och tidigare regeringars ...    3.0  
1  Kom gärna med förslag på olika samband mellan ...    3.0  
2  Sossarnas invandringspolitik bidrar till ännu ...    2.0  
3  Sossarna står ju främst för att dom vill ha et...    2.0  
4  Den nya invandrarklassen har fått det mycket s...    2.0  


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

In [37]:
###############################################################################
# metrics
def metrics(p):
    preds, gold = p.predictions.squeeze(), p.label_ids
    alpha = krippendorff.alpha(reliability_data=[gold, preds],
level_of_measurement="interval")
    return {"spearman": spearmanr(preds, gold)[0],
            "pearson": pearsonr(preds, gold)[0],
            "krippendorff_alpha": alpha}


In [7]:
# Freeze embedding layers
for param in reg_model.distilbert.embeddings.parameters():
    param.requires_grad = False

# Freeze transformer layers 0–3
for layer in reg_model.distilbert.transformer.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

# Layers 4 and 5 remain trainable
# Layers 4 and 5 remain trainable

In [8]:
for name, param in reg_model.named_parameters():
    print(name, param.requires_grad)

distilbert.embeddings.word_embeddings.weight False
distilbert.embeddings.position_embeddings.weight False
distilbert.embeddings.LayerNorm.weight False
distilbert.embeddings.LayerNorm.bias False
distilbert.transformer.layer.0.attention.q_lin.weight False
distilbert.transformer.layer.0.attention.q_lin.bias False
distilbert.transformer.layer.0.attention.k_lin.weight False
distilbert.transformer.layer.0.attention.k_lin.bias False
distilbert.transformer.layer.0.attention.v_lin.weight False
distilbert.transformer.layer.0.attention.v_lin.bias False
distilbert.transformer.layer.0.attention.out_lin.weight False
distilbert.transformer.layer.0.attention.out_lin.bias False
distilbert.transformer.layer.0.sa_layer_norm.weight False
distilbert.transformer.layer.0.sa_layer_norm.bias False
distilbert.transformer.layer.0.ffn.lin1.weight False
distilbert.transformer.layer.0.ffn.lin1.bias False
distilbert.transformer.layer.0.ffn.lin2.weight False
distilbert.transformer.layer.0.ffn.lin2.bias False
distilbe

In [9]:
args = TrainingArguments("absa-out", learning_rate=2e-5, num_train_epochs=10,
                         per_device_train_batch_size=16, weight_decay=0.01,
                         eval_strategy="epoch", save_strategy="epoch",
                         load_best_model_at_end=True,
                         metric_for_best_model="krippendorff_alpha", fp16=True)
trainer = Trainer(model=reg_model, args=args, train_dataset=tr, eval_dataset=dv,
                  processing_class=tokenizer, compute_metrics=metrics)
trainer.train()

# 4. Final evaluation on the held-out test set
print(trainer.evaluate(te, metric_key_prefix="test"))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Spearman,Pearson,Krippendorff Alpha
1,No log,5.521721,-0.253217,-0.222801,-0.761316
2,No log,3.878395,-0.169004,-0.115849,-0.684008
3,No log,2.264774,-0.107197,-0.096542,-0.524758
4,No log,1.043715,-0.124194,-0.123657,-0.206001
5,No log,0.739057,-0.086916,-0.111326,-0.035911
6,No log,0.837434,-0.067409,-0.042804,-0.083940
7,No log,0.812574,0.040175,0.029015,-0.052647
8,No log,0.762808,0.063352,0.065154,-0.008902
9,No log,0.728504,0.083053,0.086371,0.021775
10,No log,0.716956,0.088268,0.093722,0.032374


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


... (14 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Spearman,Pearson,Krippendorff Alpha
No log,0.924986,10,0.004355,-0.003165,-0.007164


{'test_loss': 0.9249855875968933, 'test_spearman': 0.004354671421994384, 'test_pearson': -0.0031653717160224915, 'test_krippendorff_alpha': -0.00716413077396072}


In [20]:
import shutil

out = "/kaggle/working/my-model"
reg_model.save_pretrained(out)        # weights (model.safetensors) + config.json
tokenizer.save_pretrained(out)    # tokenizer files

shutil.make_archive(out, "zip", out)   # creates /kaggle/working/my-model.zip

from safetensors.torch import load_file, save_file
sd = load_file("my-model/model.safetensors")
keep = {k: v.contiguous() for k, v in sd.items() if ".layer.4." in k}   # change the filter
save_file(keep, "layer4.safetensors", metadata={"format": "pt"})
print(len(keep), "tensors,", sum(v.numel() for v in keep.values()) * 4 / 1e6, "MB")
keep = {k: v.contiguous() for k, v in sd.items() if ".layer.5." in k}   # change the filter
save_file(keep, "layer5.safetensors", metadata={"format": "pt"})
print(len(keep), "tensors,", sum(v.numel() for v in keep.values()) * 4 / 1e6, "MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

16 tensors, 28.351488 MB
16 tensors, 28.351488 MB


In [28]:
import numpy as np, torch
sub = dev.sample(150, random_state=0)
enc = tokenizer(list(sub.text), truncation=True, max_length=48,
                padding="max_length", return_tensors="pt")
reg_model.eval()
dev_ = next(reg_model.parameters()).device
with torch.no_grad():
    out = reg_model(**{k: v.to(dev_) for k, v in enc.items()}, output_hidden_states=True)
g = lambda t: t.detach().cpu().numpy()
np.savez("probe_data.npz",
    h_in4=g(out.hidden_states[4]).astype(np.float16),         # input to layer 4
    h_out5_first10=g(out.hidden_states[6][:10]).astype(np.float16),  # to verify my numpy forward pass
    mask=enc["attention_mask"].numpy().astype(np.uint8),
    labels=sub.label.values.astype(np.float32),
    logits=g(out.logits).squeeze(-1).astype(np.float32),
    pre_w=g(reg_model.pre_classifier.weight), pre_b=g(reg_model.pre_classifier.bias),
    cls_w=g(reg_model.classifier.weight), cls_b=g(reg_model.classifier.bias))

In [33]:
import glob, torch
from safetensors.torch import load_file

def reinsert(model, path):
    """Copy every tensor in a .safetensors file into the model, checking names and shapes first."""
    sd = load_file(path)
    params = dict(model.named_parameters())
    for k, v in sd.items():
        if k not in params:
            raise KeyError(f"{k} is not a parameter of this model")
        if params[k].shape != v.shape:
            raise ValueError(f"{k}: file has {tuple(v.shape)}, model has {tuple(params[k].shape)}")
    with torch.no_grad():
        for k, v in sd.items():
            params[k].copy_(v.to(params[k].dtype).to(params[k].device))
    print(f"inserted {len(sd)} tensors from {path.split('/')[-1]}")
    return list(sd)

# 1) Find the file (Kaggle puts uploads inside version folders)
files = glob.glob("/kaggle/input/**/head_edited*.safetensors", recursive=True)
print(files)                      # if empty: !ls -R /kaggle/input
path = files[0]

# 2) Insert it. Use the SAME model your layers 4 and 5 were exported from.
before = reg_model.classifier.bias.detach().clone()
reinsert(reg_model, path)
reg_model.eval()

# 3) Confirm it took effect
print("classifier.bias before:", before.item(), "after:", reg_model.classifier.bias.item())

# 4) Quick check on a few paragraphs: predictions should now spread around the label scale (about 1 to 5)
import numpy as np
texts = list(sub["text"][:8]) if "text" in sub else list(sub.text[:8])
enc = tokenizer(texts, truncation=True, max_length=48, padding="max_length", return_tensors="pt")
with torch.no_grad():
    out = reg_model(**{k: v.to(next(reg_model.parameters()).device) for k, v in enc.items()})
print(np.round(out.logits.squeeze(-1).cpu().numpy(), 2))

# 5) Save the full edited model so you can reload it later
reg_model.save_pretrained("/kaggle/working/edited-model")
tokenizer.save_pretrained("/kaggle/working/edited-model")

['/kaggle/input/models/leejollans/headedit-distilbert/pytorch/default/1/head_edited.safetensors']
inserted 4 tensors from head_edited.safetensors
classifier.bias before: 1.851776123046875 after: 1.851776123046875
[2.17 2.58 2.5  2.74 2.52 2.76 2.31 2.85]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/edited-model/tokenizer_config.json',
 '/kaggle/working/edited-model/tokenizer.json')

In [38]:
args = TrainingArguments("absa-out", learning_rate=2e-5, num_train_epochs=10,
                         per_device_train_batch_size=16, weight_decay=0.01,
                         eval_strategy="epoch", save_strategy="epoch",
                         load_best_model_at_end=True,
                         metric_for_best_model="krippendorff_alpha", fp16=True)
trainer = Trainer(model=reg_model, args=args, train_dataset=tr, eval_dataset=dv,
                  processing_class=tokenizer, compute_metrics=metrics)
print(trainer.evaluate(te, metric_key_prefix="test"))

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Spearman,Pearson,Krippendorff Alpha
No log,1.019465,0,0.019419,0.038863,-0.030258


{'test_loss': 1.0194649696350098, 'test_spearman': 0.01941931706948329, 'test_pearson': 0.03886289522051811, 'test_krippendorff_alpha': -0.030257819855837642}
